# Data Preprocessing
This notebook converts relational sales and operational data into a clean, machine-learning-ready dataset for a **one-month-ahead product risk model**.

## Business objective and unit of analysis

The model should answer:

> Based on all information available at the end of the current month, which products are likely to show medium or high commercial risk next month?

| Item | Definition |
|---|---|
| Observation grain | One row per `product_id × region × year_month` |
| Prediction point | End of the current month |
| Prediction horizon | One month ahead |
| Target | `next_month_risk_label`: 0 = low, 1 = medium, 2 = high |
| Split strategy | Chronological 70% train / 15% validation / 15% test |

This timing assumption matters. Current-month sales, CRM, pipeline, inventory and market information may be used only because the prediction is assumed to run **after the month closes**.

## 1. Setup

In [5]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    # keeps the notebook testable in a plain Python environment
    display = print

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

RANDOM_SEED = 42
TARGET_COL = "next_month_risk_label"
ID_COLS = ["product_id", "year_month", "region"]

INPUT_DIR = Path("../data/sql_input")
OUTPUT_DIR = Path("../data/preprocessed_data")
SAMPLE_DIR = Path("../data/03_preprocessed_data")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input folder : {INPUT_DIR.resolve()}")
print(f"Output folder: {OUTPUT_DIR.resolve()}")
print(f"Sample folder: {SAMPLE_DIR.resolve()}")


Input folder : C:\Users\Anast\OneDrive\Desktop\AS Portfolio\early-warning\data\sql_input
Output folder: C:\Users\Anast\OneDrive\Desktop\AS Portfolio\early-warning\data\preprocessed_data
Sample folder: C:\Users\Anast\OneDrive\Desktop\AS Portfolio\early-warning\data\03_preprocessed_data


## 2. Load and inspect the source tables

In [ ]:
CSV_FILES = {
    "sales": "sales.csv",
    "products": "products.csv",
    "customers": "customers.csv",
    "regions": "regions.csv",
    "inventory": "inventory.csv",
    "costs": "costs.csv",
    "returns": "returns.csv",
    "crm": "crm_activities.csv",
    "pipeline": "pipeline.csv",
    "sales_reps": "sales_reps.csv",
    "date": "dates.csv"}

# Load CSV files 
def load_csv_tables(input_dir: Path, csv_files: dict[str, str]) -> dict[str, pd.DataFrame]:
    missing_files = [
        filename for filename in csv_files.values()
        if not (input_dir / filename).exists()]
    if missing_files:
        raise FileNotFoundError(
            "The following input files are missing from "
            f"{input_dir.resolve()}: {missing_files}")

    return {
        table_name: pd.read_csv(input_dir / filename)
        for table_name, filename in csv_files.items()}


raw = load_csv_tables(INPUT_DIR, CSV_FILES)
print(f"Loaded {len(raw)} source tables.")

Loaded 11 source tables.


In [ ]:
# inspect the table grain at a high level:
# - row and column counts reveal unexpectedly empty or very wide tables
# - duplicate row counts can indicate repeated exports
# - missing-cell percentages
# - memory usage helps explain later performance decisions

def summarize_tables(datasets: dict[str, pd.DataFrame]) -> pd.DataFrame:
    rows = []
    for name, df in datasets.items():
        rows.append({
            "table": name,
            "rows": len(df),
            "columns": df.shape[1],
            "duplicate_rows": int(df.duplicated().sum()),
            "missing_cells_pct": round(df.isna().mean().mean() * 100, 2),
            "memory_mb": round(df.memory_usage(deep=True).sum() / 1_048_576, 2)})
    return pd.DataFrame(rows).sort_values("table").reset_index(drop=True)

raw_overview = summarize_tables(raw)
display(raw_overview)

,table,rows,columns,duplicate_rows,missing_cells_pct,memory_mb
0,costs,12000,7,0,0.000,1.220
1,crm,35000,10,0,0.000,7.520
2,customers,1200,9,0,0.000,0.380
3,date,4018,18,0,0.000,1.990
4,inventory,69852,10,0,0.000,8.260
5,pipeline,15000,14,0,0.000,4.580
6,products,200,14,0,0.000,0.080
7,regions,10,7,0,0.000,0.000
8,returns,4200,10,0,0.000,0.740
9,sales,120000,26,0,0.100,57.750


In [ ]:
# A small sample for quick overview
for table_name in ["sales", "products", "inventory", "crm", "pipeline"]:
    print(f"\n{table_name.upper()} — first 3 rows")
    display(raw[table_name].head(3))


SALES — first 3 rows


,sales_id,order_id,date_id,order_date,year_month,customer_id,product_id,region_id,sales_rep_id,units,asp_eur,discount_pct,revenue_eur,revenue_local_currency,currency,unit_cost_eur,gross_profit_eur,gross_margin_pct,is_outlier_order,cogs_eur,discount_value_eur,margin_category,order_size_category,missing_sales_rep_flag,missing_discount_flag,missing_margin_flag
0,1,ORD-00000001,20220304,2022-03-04,2022-03-01,282,1133,2,39.000,26,104.000,0.092,"2,704.130","2,704.130",EUR,78.470,663.790,0.245,False,"2,040.340",248.780,Low Margin,Medium Order,0,0,0
1,2,ORD-00000002,20240503,2024-05-03,2024-05-01,562,1105,2,73.000,21,"1,698.070",0.089,"35,659.450","35,659.450",EUR,944.570,"15,823.420",0.444,False,"19,836.030","3,173.691",High Margin,Small Order,0,0,0
2,3,ORD-00000003,20241205,2024-12-05,2024-12-01,146,1007,7,41.000,117,983.150,0.123,"115,029.000","169,160.290",CAD,637.920,"40,392.640",0.351,False,"74,636.360","14,148.567",Medium Margin,Large Order,0,0,0



PRODUCTS — first 3 rows


,product_id,sku,product_family,product_group,product_name,launch_year,lifecycle_stage,base_list_price_eur,base_unit_cost_eur,target_margin_pct,product_growth_factor,is_declining_product,is_new_product,launch_date
0,1000,SKU-1000,IT Devices,Laptop Pro,Laptop Pro Model 01,2022,Decline,804.370,540.920,0.290,1.080,1,0,2022-01-01
1,1001,SKU-1001,IT Devices,Laptop Pro,Laptop Pro Model 02,2023,Growth,771.330,542.710,0.290,1.080,0,0,2023-01-01
2,1002,SKU-1002,IT Devices,Laptop Pro,Laptop Pro Model 03,2024,Growth,828.800,552.000,0.290,1.080,0,0,2024-01-01



INVENTORY — first 3 rows


,inventory_id,year_month,product_id,region_id,opening_stock_units,production_units,ending_stock_units,stockout_flag,inventory_value_eur,zero_stock_flag
0,1,2021-01-01,1000,1,53,79,51,False,"30,108.390",0
1,2,2021-01-01,1000,3,31,22,34,False,"11,838.230",0
2,3,2021-01-01,1000,7,15,13,18,False,"7,186.070",0



CRM — first 3 rows


,activity_id,date_id,activity_date,customer_id,sales_rep_id,activity_type,activity_minutes,sentiment_score,customer_health_score,customer_health_band
0,1,20241105,2024-11-05,1128,57,Call,22,1.386,90.300,Healthy
1,2,20221123,2022-11-23,1049,60,Email,40,-0.289,50.900,Neutral
2,3,20240619,2024-06-19,1006,48,Email,18,0.337,73.700,Neutral



PIPELINE — first 3 rows


,opportunity_id,created_date,customer_id,product_group,sales_rep_id,stage,expected_value_eur,win_probability,expected_close_date,created_date_id,weighted_pipeline_eur,days_to_close,is_closed_won,is_closed_lost
0,1,2021-08-12,56,Connectivity Module,18,Lead,"137,697.630",0.413,2022-03-15,20210812,"56,869.120",215,0,0
1,2,2021-02-14,1038,Accessory Kit,41,Qualified,"109,985.570",0.452,2021-07-09,20210214,"49,713.480",145,0,0
2,3,2022-07-09,546,Service Contract,59,Qualified,"31,257.790",0.153,2022-12-06,20220709,"4,782.440",150,0,0


## 3. Validate the source schema and relational keys

A professional pipeline should fail early when an upstream table changes. The validation below lists every field that is genuinely required by this notebook.

Fields that are merely useful enrichment attributes remain optional and are selected only when present.


In [ ]:
REQUIRED_SOURCE_COLUMNS = {
    "sales": [
        "order_date", "customer_id", "product_id", "region_id",
        "sales_rep_id", "units", "revenue_eur", "discount_pct"],
    "products": ["product_id", "product_family", "product_group"],
    "customers": ["customer_id", "region_id"],
    "regions": ["region_id", "region"],
    "sales_reps": ["sales_rep_id", "region_id"],
    "inventory": [
        "year_month", "product_id", "region_id", "opening_stock_units",
        "production_units", "ending_stock_units", "stockout_flag",
        "zero_stock_flag", "inventory_value_eur"],
    "costs": [
        "year_month", "product_id", "standard_unit_cost_eur",
        "actual_unit_cost_eur", "cost_variance_eur", "cost_variance_pct"],
    "returns": [
        "return_id", "return_date", "product_id", "region_id",
        "return_units", "return_value_eur"],
    "crm": [
        "activity_id", "activity_date", "customer_id", "activity_type",
        "activity_minutes", "sentiment_score", "customer_health_score"],
    "pipeline": [
        "opportunity_id", "created_date", "customer_id", "product_group",
        "weighted_pipeline_eur", "expected_value_eur", "win_probability",
        "is_closed_won", "is_closed_lost"],
    "date": []
}

# List missing columns by table
def validate_required_columns(
    datasets: dict[str, pd.DataFrame],
    required_columns: dict[str, list[str]],
) -> pd.DataFrame:
    issues = []
    for table_name, columns in required_columns.items():
        if table_name not in datasets:
            issues.append({"table": table_name, "missing_column": "<table missing>"})
            continue
        for column in columns:
            if column not in datasets[table_name].columns:
                issues.append({"table": table_name, "missing_column": column})

    # These tables accept one of two alternative date/ID columns.
    alternatives = {
        "sales transaction ID": ("sales", {"sales_id", "order_id"}),
        "date dimension date": ("date", {"MonthStartDate", "Date"})}
    for label, (table_name, options) in alternatives.items():
        if not options.intersection(datasets[table_name].columns):
            issues.append({
                "table": table_name,
                "missing_column": f"{label}: one of {sorted(options)}"})

    return pd.DataFrame(issues, columns=["table", "missing_column"])


schema_issues = validate_required_columns(raw, REQUIRED_SOURCE_COLUMNS)
if not schema_issues.empty:
    display(schema_issues)
    raise ValueError("Source schema validation failed. See the table above.")

print("Schema validation passed: all required fields are available.")

Schema validation passed: all required fields are available.


In [12]:
# Standardization helpers 

# Create stable string join keys from integer-, float- or string-like IDs
def normalize_id(series: pd.Series) -> pd.Series:
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .replace({"<NA>": pd.NA, "nan": pd.NA, "None": pd.NA, "": pd.NA}))

# Convert dates or YYYY-MM values to month-start timestamps
def to_month(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, errors="coerce").dt.to_period("M").dt.to_timestamp()

# Return a numeric version of a column or a default series when it is optional
def numeric(df: pd.DataFrame, column: str, default: float = np.nan) -> pd.Series:
    if column not in df.columns:
        return pd.Series(default, index=df.index, dtype="float64")
    return pd.to_numeric(df[column], errors="coerce")

# Keep only requested columns that actually exist
def existing(df: pd.DataFrame, columns: list[str]) -> list[str]:
    return [column for column in columns if column in df.columns]

## 4. Standardize dimension tables

In [13]:
products = raw["products"].copy()
products["product_id"] = normalize_id(products["product_id"])
products["product_line"] = products["product_family"].fillna("Unknown")
products["product_category"] = products["product_group"].fillna("Unknown")
products["launch_date"] = (
    pd.to_datetime(products["launch_date"], errors="coerce")
    if "launch_date" in products
    else pd.NaT)
products = products.drop_duplicates("product_id").reset_index(drop=True)

regions = raw["regions"].copy()
regions["region_id"] = normalize_id(regions["region_id"])
regions = regions.drop_duplicates("region_id").reset_index(drop=True)

customers = raw["customers"].copy()
customers["customer_id"] = normalize_id(customers["customer_id"])
customers["region_id"] = normalize_id(customers["region_id"])
customers = customers.drop_duplicates("customer_id").reset_index(drop=True)

sales_reps = raw["sales_reps"].copy()
sales_reps["sales_rep_id"] = normalize_id(sales_reps["sales_rep_id"])
sales_reps["region_id"] = normalize_id(sales_reps["region_id"])
sales_reps = sales_reps.drop_duplicates("sales_rep_id").reset_index(drop=True)

product_dim_cols = existing(products, [
    "product_id", "product_line", "product_category", "product_name", "sku",
    "lifecycle_stage", "launch_date", "is_declining_product", "is_new_product",
    "base_list_price_eur", "base_unit_cost_eur", "target_margin_pct",
    "product_growth_factor"])
region_dim_cols = existing(regions, [
    "region_id", "region", "country", "currency", "fx_to_eur",
    "market_growth_factor", "margin_factor"])

In [14]:
# Check dimension uniqueness and coverage
dimension_check = pd.DataFrame([
    {
        "dimension": "products",
        "rows": len(products),
        "unique_key_values": products["product_id"].nunique(dropna=True),
        "missing_keys": int(products["product_id"].isna().sum())},
    {
        "dimension": "customers",
        "rows": len(customers),
        "unique_key_values": customers["customer_id"].nunique(dropna=True),
        "missing_keys": int(customers["customer_id"].isna().sum()),
    },
    {
        "dimension": "regions",
        "rows": len(regions),
        "unique_key_values": regions["region_id"].nunique(dropna=True),
        "missing_keys": int(regions["region_id"].isna().sum()),
    },
    {
        "dimension": "sales_reps",
        "rows": len(sales_reps),
        "unique_key_values": sales_reps["sales_rep_id"].nunique(dropna=True),
        "missing_keys": int(sales_reps["sales_rep_id"].isna().sum())}])
dimension_check["key_is_unique"] = (
    dimension_check["rows"] == dimension_check["unique_key_values"]
) & dimension_check["missing_keys"].eq(0)
display(dimension_check)

,dimension,rows,unique_key_values,missing_keys,key_is_unique
0,products,200,200,0,True
1,customers,1200,1200,0,True
2,regions,10,10,0,True
3,sales_reps,85,85,0,True


## 5. Create canonical sales transaction table

One consistent naming convention:

- `units` → `units_sold`
- `revenue_eur` → `revenue`
- dates → `year_month`

In [15]:
# Product, region, and sales-representative attributes are merged with `validate="m:1"`
# An error is raised if a supposedly unique dimension key unexpectedly matches multiple rows
sales = raw["sales"].copy()

for column in ["product_id", "customer_id", "region_id", "sales_rep_id"]:
    sales[column] = normalize_id(sales[column])

sales["year_month"] = to_month(sales["order_date"])
sales["units_sold"] = numeric(sales, "units")
sales["revenue"] = numeric(sales, "revenue_eur")
sales["avg_discount_pct"] = numeric(sales, "discount_pct")
sales["transaction_id"] = normalize_id(
    sales["sales_id"] if "sales_id" in sales else sales["order_id"])

# Remove any duplicate descriptive columns before applying canonical dimensions
sales = sales.drop(
    columns=[
        column for column in product_dim_cols + region_dim_cols
        if column not in {"product_id", "region_id"} and column in sales.columns],
    errors="ignore")
sales = sales.merge(
    regions[region_dim_cols],
    on="region_id",
    how="left",
    validate="m:1")
sales = sales.merge(
    products[product_dim_cols],
    on="product_id",
    how="left",
    validate="m:1")

rep_cols = existing(sales_reps, ["sales_rep_id", "seniority", "annual_quota_eur"])
sales = sales.drop(
    columns=[column for column in rep_cols if column != "sales_rep_id" and column in sales],
    errors="ignore")
sales = sales.merge(
    sales_reps[rep_cols],
    on="sales_rep_id",
    how="left",
    validate="m:1")

sales["region"] = sales["region"].fillna("Unknown")
sales["product_line"] = sales["product_line"].fillna("Unknown")
sales["product_category"] = sales["product_category"].fillna("Unknown")

invalid_sales_rows = (
    sales["year_month"].isna()
    | sales["product_id"].isna()
    | sales["region_id"].isna()
    | sales["transaction_id"].isna())
rejected_sales = sales.loc[invalid_sales_rows].copy()
sales = sales.loc[~invalid_sales_rows].reset_index(drop=True)

print(f"Valid canonical transactions: {len(sales):,}")
print(f"Rejected transactions: {len(rejected_sales):,}")


Valid canonical transactions: 120,000
Rejected transactions: 0


In [16]:
#Check canonical sales quality

sales_quality = pd.Series({
    "rows": len(sales),
    "unique_transactions": sales["transaction_id"].nunique(),
    "start_month": sales["year_month"].min(),
    "end_month": sales["year_month"].max(),
    "unknown_products": int((sales["product_line"] == "Unknown").sum()),
    "unknown_regions": int((sales["region"] == "Unknown").sum()),
    "missing_customers": int(sales["customer_id"].isna().sum()),
    "negative_units": int((sales["units_sold"] < 0).sum()),
    "negative_revenue": int((sales["revenue"] < 0).sum())}, name="value")
display(sales_quality.to_frame())
display(sales.head(5))

,value
rows,120000
unique_transactions,120000
start_month,2021-01-01 00:00:00
end_month,2025-12-01 00:00:00
unknown_products,0
unknown_regions,0
missing_customers,0
negative_units,0
negative_revenue,0


,sales_id,order_id,date_id,order_date,year_month,customer_id,product_id,region_id,sales_rep_id,units,asp_eur,discount_pct,revenue_eur,revenue_local_currency,unit_cost_eur,gross_profit_eur,gross_margin_pct,is_outlier_order,cogs_eur,discount_value_eur,margin_category,order_size_category,missing_sales_rep_flag,missing_discount_flag,missing_margin_flag,units_sold,revenue,avg_discount_pct,transaction_id,region,country,currency,fx_to_eur,market_growth_factor,margin_factor,product_line,product_category,product_name,sku,lifecycle_stage,launch_date,is_declining_product,is_new_product,base_list_price_eur,base_unit_cost_eur,target_margin_pct,product_growth_factor,seniority,annual_quota_eur
0,1,ORD-00000001,20220304,2022-03-04,2022-03-01,282,1133,2,39,26,104.000,0.092,"2,704.130","2,704.130",78.470,663.790,0.245,False,"2,040.340",248.780,Low Margin,Medium Order,0,0,0,26,"2,704.130",0.092,1,Western Europe,France,EUR,1.000,1.020,0.970,Components,Connectivity Module,Connectivity Module Model 14,SKU-1133,Growth,2019-01-01,0,0,106.410,75.770,0.310,1.180,Professional,"1,229,114.000"
1,2,ORD-00000002,20240503,2024-05-03,2024-05-01,562,1105,2,73,21,"1,698.070",0.089,"35,659.450","35,659.450",944.570,"15,823.420",0.444,False,"19,836.030","3,173.691",High Margin,Small Order,0,0,0,21,"35,659.450",0.089,2,Western Europe,France,EUR,1.000,1.020,0.970,Industrial Tech,Industrial Scanner,Industrial Scanner Model 06,SKU-1105,New,2023-01-01,0,1,"1,496.620",906.260,0.340,1.030,Professional,"1,352,559.000"
2,3,ORD-00000003,20241205,2024-12-05,2024-12-01,146,1007,7,41,117,983.150,0.123,"115,029.000","169,160.290",637.920,"40,392.640",0.351,False,"74,636.360","14,148.567",Medium Margin,Large Order,0,0,0,117,"115,029.000",0.123,3,North America,Canada,CAD,0.680,1.040,0.990,IT Devices,Laptop Pro,Laptop Pro Model 08,SKU-1007,Growth,2024-01-01,0,0,"1,017.790",613.650,0.290,1.080,Professional,"1,276,718.000"
3,4,ORD-00000004,20241119,2024-11-19,2024-11-01,487,1176,6,10,5,63.480,0.084,317.390,344.990,55.670,39.040,0.123,False,278.350,26.661,Low Margin,Small Order,0,0,0,5,317.390,0.084,4,North America,United States,USD,0.920,1.140,1.030,Accessories,Accessory Kit,Accessory Kit Model 17,SKU-1176,Growth,2018-01-01,0,0,61.530,51.180,0.220,1.050,Professional,"1,217,284.000"
4,5,ORD-00000005,20220713,2022-07-13,2022-07-01,869,1104,10,81,61,931.870,0.157,"56,844.250","9,168,427.420",728.640,"12,397.020",0.218,False,"44,447.230","8,924.547",Low Margin,Medium Order,0,0,0,61,"56,844.250",0.157,5,APAC,Japan,JPY,0.006,0.950,1.020,Industrial Tech,Industrial Scanner,Industrial Scanner Model 05,SKU-1104,Growth,2023-01-01,0,0,998.190,692.650,0.340,1.030,Professional,"1,311,256.000"


In [17]:
# Referential-integrity checks quantify unmatched fact-table keys
relationship_checks = pd.Series({
    "sales_unknown_products": int(
        (~sales["product_id"].isin(products["product_id"])).sum()),
    "sales_unknown_customers": int(
        (~sales["customer_id"].isin(customers["customer_id"])).sum()),
    "sales_unknown_regions": int(
        (~sales["region_id"].isin(regions["region_id"])).sum()),
    "sales_unknown_reps": int(
        (~sales["sales_rep_id"].isin(sales_reps["sales_rep_id"])).sum()),
}, name="unmatched_rows")
display(relationship_checks.to_frame())

,unmatched_rows
sales_unknown_products,0
sales_unknown_customers,0
sales_unknown_regions,0
sales_unknown_reps,730


## 6. Aggregate sales to the modeling grain
Later we want to model predictions not on individual transactions. We therefore aggregate sales to **product × region × month**.

We also retain a customer–product–month table as a reusable analytical output. It can support customer-level drillthrough or recommendation use cases without changing the EWS model grain.

In [18]:
customer_product_monthly = (
    sales.groupby(
        ["year_month", "customer_id", "product_id"],
        dropna=False,
        as_index=False)
    .agg(
        units_sold=("units_sold", "sum"),
        revenue=("revenue", "sum"),
        order_count=("transaction_id", "nunique"),
        avg_discount_pct=("avg_discount_pct", "mean")))

monthly_aggregation = {
    "units_sold": ("units_sold", "sum"),
    "revenue": ("revenue", "sum"),
    "unique_customers": ("customer_id", "nunique"),
    "avg_discount_pct": ("avg_discount_pct", "mean"),
    "order_count": ("transaction_id", "nunique"),
    "active_sales_reps": ("sales_rep_id", "nunique")}
for optional_measure in ["gross_profit_eur", "cogs_eur"]:
    if optional_measure in sales.columns:
        monthly_aggregation[optional_measure] = (optional_measure, "sum")

monthly_sales = (
    sales.groupby(
        ["product_id", "year_month", "region"],
        as_index=False)
    .agg(**monthly_aggregation))

# Sum each active representative's quota once per product-region-month.
if "annual_quota_eur" in sales.columns:
    quota_by_month = (
        sales.dropna(subset=["sales_rep_id"])
        .drop_duplicates(
            ["product_id", "year_month", "region", "sales_rep_id"])
        .groupby(["product_id", "year_month", "region"], as_index=False)
        .agg(covered_annual_quota_eur=("annual_quota_eur", "sum")))
    monthly_sales = monthly_sales.merge(
        quota_by_month,
        on=["product_id", "year_month", "region"],
        how="left",
        validate="1:1")

In [19]:
# Check: does aggregation reconcile with the transactions?
reconciliation = pd.DataFrame({
    "measure": ["units_sold", "revenue"],
    "transaction_total": [
        sales["units_sold"].sum(),
        sales["revenue"].sum()],
    "monthly_total": [
        monthly_sales["units_sold"].sum(),
        monthly_sales["revenue"].sum()]})
reconciliation["difference"] = (
    reconciliation["monthly_total"] - reconciliation["transaction_total"])
display(reconciliation)
display(monthly_sales.head(5))

,measure,transaction_total,monthly_total,difference
0,units_sold,"4,759,689.000","4,759,689.000",0.000
1,revenue,"2,361,948,725.010","2,361,948,725.010",0.000


,product_id,year_month,region,units_sold,revenue,unique_customers,avg_discount_pct,order_count,active_sales_reps,gross_profit_eur,cogs_eur,covered_annual_quota_eur
0,1000,2021-01-01,DACH,81,"56,358.020",2,0.084,2,2,"12,852.590","43,505.430","3,446,113.000"
1,1000,2021-01-01,North America,10,"6,669.060",1,0.132,1,1,"1,233.940","5,435.120","2,615,971.000"
2,1000,2021-01-01,Southern Europe,19,"12,332.030",1,0.088,1,1,"2,069.160","10,262.870","1,376,181.000"
3,1000,2021-02-01,North America,7,"4,708.680",1,0.058,1,1,636.350,"4,072.330","2,761,520.000"
4,1000,2021-02-01,Northern Europe,14,"9,416.400",1,0.084,1,1,"1,534.140","7,882.260","2,091,903.000"


## 7. Build a complete product–region–month panel

A sales fact table normally omits months with zero transactions. For an early-warning model, those missing months are meaningful observations, not missing data.

The complete panel makes zero-sales months explicit and allows lags and rolling windows to represent real month-to-month behavior. Months before a known product launch are excluded.


In [21]:
date_candidates = [sales["year_month"]]
for table_name in ["inventory", "costs"]:
    if "year_month" in raw[table_name]:
        date_candidates.append(to_month(raw[table_name]["year_month"]))

all_months_observed = pd.concat(date_candidates, ignore_index=True).dropna()
min_month = all_months_observed.min()
max_month = all_months_observed.max()
months = pd.date_range(min_month, max_month, freq="MS")

product_ids = pd.Index(products["product_id"].dropna().unique()).union(
    pd.Index(sales["product_id"].dropna().unique()))
region_names = pd.Index(regions["region"].dropna().unique())

panel = (
    pd.MultiIndex.from_product(
        [product_ids, months, region_names],
        names=["product_id", "year_month", "region"])
    .to_frame(index=False)
    .merge(
        products[product_dim_cols],
        on="product_id",
        how="left",
        validate="m:1"))

panel["product_line"] = panel["product_line"].fillna("Unknown")
panel["product_category"] = panel["product_category"].fillna("Unknown")

launch_month = panel["launch_date"].dt.to_period("M").dt.to_timestamp()
panel = panel.loc[
    panel["launch_date"].isna() | (panel["year_month"] >= launch_month)
].copy()

modeling = panel.merge(
    monthly_sales,
    on=["product_id", "year_month", "region"],
    how="left",
    validate="1:1")

zero_when_no_sales = [
    "units_sold", "revenue", "unique_customers", "order_count",
    "active_sales_reps"]
zero_when_no_sales += existing(
    modeling,
    ["gross_profit_eur", "cogs_eur", "covered_annual_quota_eur"])
modeling[zero_when_no_sales] = modeling[zero_when_no_sales].fillna(0)

# A discount of zero is not assumed when no transaction exists.
# Leaving it missing allows train-fitted imputation to handle it later.

print(
    f"Complete panel: {len(modeling):,} rows across "
    f"{len(months)} months, {len(product_ids)} products "
    f"and {len(region_names)} regions.")


Complete panel: 78,816 rows across 60 months, 200 products and 8 regions.


In [ ]:
# Check zero-sales months and panel uniqueness
panel_check = pd.Series({
    "panel_rows": len(modeling),
    "duplicate_panel_keys": int(modeling.duplicated(["product_id", "year_month", "region"]).sum()),
    "zero_sales_rows": int(modeling["units_sold"].eq(0).sum()),
    "zero_sales_share_pct": round(modeling["units_sold"].eq(0).mean() * 100, 2),
    "start_month": modeling["year_month"].min(),
    "end_month": modeling["year_month"].max()}, name="value")
display(panel_check.to_frame())

example_panel = (
    modeling.sort_values(["product_id", "region", "year_month"])
    .loc[:, ["product_id", "region", "year_month", "units_sold", "revenue"]]
    .head(12))
display(example_panel)

,value
panel_rows,78816
duplicate_panel_keys,0
zero_sales_rows,27984
zero_sales_share_pct,35.510
start_month,2021-01-01 00:00:00
end_month,2025-12-01 00:00:00


,product_id,region,year_month,units_sold,revenue
7,1000,APAC,2022-01-01,0.000,0.000
15,1000,APAC,2022-02-01,113.000,"70,747.880"
23,1000,APAC,2022-03-01,0.000,0.000
31,1000,APAC,2022-04-01,0.000,0.000
39,1000,APAC,2022-05-01,69.000,"47,046.420"
47,1000,APAC,2022-06-01,30.000,"18,141.870"
55,1000,APAC,2022-07-01,0.000,0.000
63,1000,APAC,2022-08-01,0.000,0.000
71,1000,APAC,2022-09-01,0.000,0.000
79,1000,APAC,2022-10-01,24.000,"17,019.910"


## 8. Add operational and commercial context
Enrich the monthly panel with information that may explain sales risk:

- inventory and stockouts,
- costs and returns,
- calendar attributes,
- CRM engagement,
- sales pipeline activity.

Each source is first aggregated to a compatible grain. Merge validation prevents accidental row multiplication.

In [25]:
inventory = raw["inventory"].copy()
inventory["product_id"] = normalize_id(inventory["product_id"])
inventory["region_id"] = normalize_id(inventory["region_id"])
inventory["year_month"] = to_month(inventory["year_month"])
inventory["stockout_flag"] = numeric(inventory, "stockout_flag", 0).fillna(0).astype(int)
inventory["zero_stock_flag"] = numeric(inventory, "zero_stock_flag", 0).fillna(0).astype(int)

inventory = inventory.merge(
    regions[["region_id", "region"]],
    on="region_id",
    how="left",
    validate="m:1")
inventory_monthly = (
    inventory.groupby(
        ["product_id", "year_month", "region"],
        as_index=False)
    .agg(
        opening_stock_units=("opening_stock_units", "sum"),
        production_units=("production_units", "sum"),
        ending_stock_units=("ending_stock_units", "sum"),
        stockout_flag=("stockout_flag", "max"),
        zero_stock_flag=("zero_stock_flag", "max"),
        inventory_value_eur=("inventory_value_eur", "sum")))

modeling = modeling.merge(
    inventory_monthly,
    on=["product_id", "year_month", "region"],
    how="left",
    validate="1:1")

inventory_available = (
    modeling["opening_stock_units"].notna()
    | modeling["production_units"].notna())
available_supply = (
    modeling["opening_stock_units"].fillna(0)
    + modeling["production_units"].fillna(0))
modeling["backorder_units"] = np.where(
    inventory_available,
    (modeling["units_sold"] - available_supply).clip(lower=0),
    0)
modeling["stockout_flag"] = modeling["stockout_flag"].fillna(0).astype(int)
modeling["zero_stock_flag"] = modeling["zero_stock_flag"].fillna(0).astype(int)

In [26]:
costs = raw["costs"].copy()
costs["product_id"] = normalize_id(costs["product_id"])
costs["year_month"] = to_month(costs["year_month"])
cost_monthly = (
    costs.groupby(["product_id", "year_month"], as_index=False)
    .agg(
        standard_unit_cost_eur=("standard_unit_cost_eur", "mean"),
        actual_unit_cost_eur=("actual_unit_cost_eur", "mean"),
        cost_variance_eur=("cost_variance_eur", "mean"),
        cost_variance_pct=("cost_variance_pct", "mean")))
modeling = modeling.merge(
    cost_monthly,
    on=["product_id", "year_month"],
    how="left",
    validate="m:1")

returns = raw["returns"].copy()
returns["product_id"] = normalize_id(returns["product_id"])
returns["region_id"] = normalize_id(returns["region_id"])
returns["year_month"] = to_month(returns["return_date"])
returns = returns.merge(
    regions[["region_id", "region"]],
    on="region_id",
    how="left",
    validate="m:1")
returns_monthly = (
    returns.groupby(
        ["product_id", "year_month", "region"],
        as_index=False)
    .agg(
        return_units=("return_units", "sum"),
        return_value_eur=("return_value_eur", "sum"),
        return_count=("return_id", "nunique")))
modeling = modeling.merge(
    returns_monthly,
    on=["product_id", "year_month", "region"],
    how="left",
    validate="1:1")
return_columns = ["return_units", "return_value_eur", "return_count"]
modeling[return_columns] = modeling[return_columns].fillna(0)
modeling["return_rate_units"] = (
    modeling["return_units"] / modeling["units_sold"].replace(0, np.nan))

dates = raw["date"].copy()
source_date_column = "MonthStartDate" if "MonthStartDate" in dates else "Date"
dates["year_month"] = to_month(dates[source_date_column])
date_monthly = dates.sort_values("year_month").drop_duplicates("year_month")
calendar_cols = existing(
    date_monthly,
    ["year_month", "Year", "Quarter", "QuarterName", "Month", "MonthName", "YearMonthKey"])
modeling = modeling.merge(
    date_monthly[calendar_cols],
    on="year_month",
    how="left",
    validate="m:1")

In [27]:
# Map each customer to its canonical region before aggregating CRM activity
customer_regions = customers[["customer_id", "region_id"]].rename(
    columns={"region_id": "customer_region_id"})
region_lookup = regions[["region_id", "region"]].rename(
    columns={"region_id": "customer_region_id"})

crm = raw["crm"].copy()
crm["customer_id"] = normalize_id(crm["customer_id"])
crm["year_month"] = to_month(crm["activity_date"])
crm = crm.drop(columns=["region_id", "region"], errors="ignore")
crm = crm.merge(
    customer_regions,
    on="customer_id",
    how="left",
    validate="m:1")
crm = crm.merge(
    region_lookup,
    on="customer_region_id",
    how="left",
    validate="m:1")
crm["is_demo_like"] = (
    crm["activity_type"]
    .astype("string")
    .str.contains("demo|visit|meeting|presentation", case=False, na=False)
    .astype(int))
crm_region_month = (
    crm.dropna(subset=["year_month", "region"])
    .groupby(["year_month", "region"], as_index=False)
    .agg(
        crm_activity_count=("activity_id", "nunique"),
        crm_activity_minutes=("activity_minutes", "sum"),
        crm_demo_like_count=("is_demo_like", "sum"),
        avg_sentiment_score=("sentiment_score", "mean"),
        avg_customer_health_score=("customer_health_score", "mean")))
modeling = modeling.merge(
    crm_region_month,
    on=["year_month", "region"],
    how="left",
    validate="m:1")

pipeline = raw["pipeline"].copy()
pipeline["customer_id"] = normalize_id(pipeline["customer_id"])
pipeline["year_month"] = to_month(pipeline["created_date"])
pipeline["product_category"] = pipeline["product_group"].fillna("Unknown")
pipeline = pipeline.drop(columns=["region_id", "region"], errors="ignore")
pipeline = pipeline.merge(
    customer_regions,
    on="customer_id",
    how="left",
    validate="m:1")
pipeline = pipeline.merge(
    region_lookup,
    on="customer_region_id",
    how="left",
    validate="m:1")
pipeline_monthly = (
    pipeline.dropna(subset=["year_month", "region"])
    .groupby(["year_month", "region", "product_category"], as_index=False)
    .agg(
        pipeline_opportunities=("opportunity_id", "nunique"),
        weighted_pipeline_eur=("weighted_pipeline_eur", "sum"),
        expected_pipeline_eur=("expected_value_eur", "sum"),
        avg_win_probability=("win_probability", "mean"),
        closed_won_count=("is_closed_won", "sum"),
        closed_lost_count=("is_closed_lost", "sum")))
modeling = modeling.merge(
    pipeline_monthly,
    on=["year_month", "region", "product_category"],
    how="left",
    validate="m:1")

activity_zero_columns = [
    "crm_activity_count", "crm_activity_minutes", "crm_demo_like_count",
    "pipeline_opportunities", "weighted_pipeline_eur",
    "expected_pipeline_eur", "closed_won_count", "closed_lost_count"]
modeling[activity_zero_columns] = modeling[activity_zero_columns].fillna(0)

""" # These names are retained for compatibility with the original EWS schema.
# They are documented as proxies rather than literal website events.
modeling["campaign_flag"] = (
    (modeling["crm_activity_count"] > 0)
    | (modeling["pipeline_opportunities"] > 0)
).astype(int)
modeling["website_visits"] = modeling["crm_activity_count"].astype(int)
modeling["demo_requests"] = (
    modeling["crm_demo_like_count"]
    + modeling["pipeline_opportunities"]
).astype(int) """

' # These names are retained for compatibility with the original EWS schema.\n# They are documented as proxies rather than literal website events.\nmodeling["campaign_flag"] = (\n    (modeling["crm_activity_count"] > 0)\n    | (modeling["pipeline_opportunities"] > 0)\n).astype(int)\nmodeling["website_visits"] = modeling["crm_activity_count"].astype(int)\nmodeling["demo_requests"] = (\n    modeling["crm_demo_like_count"]\n    + modeling["pipeline_opportunities"]\n).astype(int) '

In [28]:
# Check: enrichment coverage
enrichment_columns = [
    "opening_stock_units", "actual_unit_cost_eur", "return_units",
    "crm_activity_count", "pipeline_opportunities"]
enrichment_coverage = pd.DataFrame({
    "column": enrichment_columns,
    "available_rows_pct": [
        round(modeling[column].notna().mean() * 100, 2)
        for column in enrichment_columns
    ],
    "non_zero_rows_pct": [
        round(modeling[column].fillna(0).ne(0).mean() * 100, 2)
        for column in enrichment_columns]})
display(enrichment_coverage)

,column,available_rows_pct,non_zero_rows_pct
0,opening_stock_units,64.490,64.430
1,actual_unit_cost_eur,100.000,100.000
2,return_units,100.000,4.330
3,crm_activity_count,100.000,100.000
4,pipeline_opportunities,100.000,90.270


## 9. Create market-context proxies

The source data does not contain literal market-demand or competitor indices. Instead of generating random values, this notebook derives transparent proxies from observed sales:

- **Market demand index:** current category–region units relative to the prior six-month mean; 100 is the baseline.
- **Competitor pressure index:** 100 minus the product's current unit share within its category and region.

These are descriptive proxies, not external market research. Their limitation is recorded in the data dictionary.
